In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys

import torch

SCENE = "HCM0204"
BRANCH = "mdd_pixel_gs"

WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "gaussian-splatting"
CLEAN_ROOT = WORK_ROOT / "cleaned_hcm0204"
PIXELGS_SMOKE_MODEL_ROOT = WORK_ROOT / "pixelgs_smoke_models"
PIXELGS_MODEL_ROOT = WORK_ROOT / "pixelgs_models"
PIXELGS_IMAGE_ROOT = WORK_ROOT / "pixelgs_images"
PIXELGS_EVAL_ROOT = WORK_ROOT / "pixelgs_evaluation"

assert torch.cuda.is_available(), "Hay bat GPU Accelerator trong Kaggle"
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
matches = [
    path
    for path in Path("/kaggle/input").rglob(SCENE)
    if (path / "train" / "images").is_dir()
    and (path / "train" / "sparse" / "0" / "cameras.bin").is_file()
    and (path / "test" / "images").is_dir()
    and (path / "test" / "test_poses.csv").is_file()
]

print("Cac scene tim thay:")
for path in matches:
    print(" -", path)

assert len(matches) == 1, (
    "Khong tim duoc duy nhat HCM0204. "
    "Hay kiem tra dataset da duoc Add Input vao notebook."
)

SCENE_SOURCE = matches[0]
PUBLIC_ROOT = SCENE_SOURCE.parent
print("HCM0204:", SCENE_SOURCE)
print("Public root:", PUBLIC_ROOT)


In [ ]:
if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "--branch", BRANCH,
            "--single-branch",
            "https://github.com/fulx17/gaussian-splatting.git",
            str(REPO),
        ],
        check=True,
    )
else:
    assert (REPO / ".git").exists(), f"{REPO} khong phai Git repository"
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO, check=True)
    subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=REPO, check=True)

subprocess.run(
    [
        "git", "submodule", "update", "--init", "--recursive",
        "submodules/diff-gaussian-rasterization",
        "submodules/simple-knn",
        "submodules/fused-ssim",
    ],
    cwd=REPO,
    check=True,
)

os.chdir(REPO)
subprocess.run(["git", "branch", "--show-current"], check=True)
subprocess.run(["git", "log", "-1", "--oneline"], check=True)


In [ ]:
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "colmap"], check=True)

subprocess.run(
    [sys.executable, "scripts/apply_improved_gs_rasterizer_patch.py"],
    cwd=REPO,
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-build-isolation",
        "--no-cache-dir",
        "--force-reinstall",
        str(REPO / "submodules" / "diff-gaussian-rasterization"),
    ],
    check=True,
)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        str(REPO / "submodules" / "simple-knn"),
        str(REPO / "submodules" / "fused-ssim"),
        "plyfile", "pycolmap", "lpips",
    ],
    check=True,
)
print("Dependencies installed")


In [ ]:
subprocess.run(
    [
        sys.executable,
        "scripts/apply_improved_gs_rasterizer_patch.py",
        "--check-only",
    ],
    cwd=REPO,
    check=True,
)

from diff_gaussian_rasterization import GaussianRasterizationSettings

print("Rasterizer fields:", GaussianRasterizationSettings._fields)
assert "pixel_weights" in GaussianRasterizationSettings._fields
assert "track_pixel_counts" in GaussianRasterizationSettings._fields

subprocess.run(
    [sys.executable, "scripts/smoke_test_improved_gs_rasterizer.py"],
    cwd=REPO,
    check=True,
)
print("Pixel-GS rasterizer hoat dong tren:", torch.cuda.get_device_name(0))


In [ ]:
os.chdir(REPO)

from preprocess import undistort_scene, preprocess_scene
from scene.colmap_loader import read_intrinsics_binary

CLEAN_SCENE = CLEAN_ROOT / SCENE
if not CLEAN_SCENE.exists():
    CLEAN_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SCENE_SOURCE, CLEAN_SCENE)
    undistort_scene(CLEAN_SCENE)
    preprocess_scene(CLEAN_SCENE)
else:
    print("Su dung lai scene da preprocess:", CLEAN_SCENE)

train_images = [
    path for path in (CLEAN_SCENE / "train" / "images").iterdir()
    if path.is_file()
]
test_images = [
    path for path in (CLEAN_SCENE / "test" / "images").iterdir()
    if path.is_file()
]
with open(CLEAN_SCENE / "test" / "test_poses.csv", newline="") as file:
    test_rows = list(csv.DictReader(file))
cameras = read_intrinsics_binary(
    str(CLEAN_SCENE / "train" / "sparse" / "0" / "cameras.bin")
)

print("Train images:", len(train_images))
print("Test GT images:", len(test_images))
print("Test poses:", len(test_rows))
print("Camera models:", [camera.model for camera in cameras.values()])
assert len(train_images) == 240
assert len(test_images) == 60
assert len(test_rows) == 60
assert all(camera.model == "PINHOLE" for camera in cameras.values())
print("Preprocessing HCM0204 hoan tat")


In [ ]:
def train_pixelgs(iterations, model_root, checkpoints):
    model_root = Path(model_root)
    ply_path = (
        model_root
        / SCENE
        / "point_cloud"
        / f"iteration_{iterations}"
        / "point_cloud.ply"
    )
    if ply_path.exists():
        print("Pixel-GS model da ton tai, bo qua train:", ply_path)
        return ply_path

    cmd = [
        sys.executable, "train_scene.py",
        "--input_dir", str(CLEAN_ROOT),
        "--model_dir", str(model_root),
        "--scene_name", SCENE,
        "--iterations", str(iterations),
        "--resolution", "1",
        "--test_iterations", "-1",
        "--save_iterations", str(iterations),
        "--density_control", "pixelgs",
        "--pixelgs_depth_threshold", "0.37",
        "--densify_grad_threshold", "0.0002",
        "--cap_max", "6000000",
        "--seed", "0",
    ]
    if checkpoints:
        cmd.extend(["--checkpoint_iterations", *map(str, checkpoints)])

    env = os.environ.copy()
    env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    print("Running Pixel-GS command:")
    print(" ".join(cmd))
    subprocess.run(cmd, cwd=REPO, env=env, check=True)
    assert ply_path.exists(), f"Khong tim thay Pixel-GS model: {ply_path}"
    return ply_path


In [ ]:
pixelgs_smoke_model = train_pixelgs(
    iterations=700,
    model_root=PIXELGS_SMOKE_MODEL_ROOT,
    checkpoints=[700],
)
print("Pixel-GS smoke training thanh cong:", pixelgs_smoke_model)


In [ ]:
pixelgs_final_model = train_pixelgs(
    iterations=30000,
    model_root=PIXELGS_MODEL_ROOT,
    checkpoints=[7000, 15000, 22500, 30000],
)
print("Pixel-GS full training thanh cong:", pixelgs_final_model)


In [ ]:
PIXELGS_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
pixelgs_render_cmd = [
    sys.executable, "render_scene.py",
    "--model_dir", str(PIXELGS_MODEL_ROOT),
    "--input_dir", str(CLEAN_ROOT),
    "--image_dir", str(PIXELGS_IMAGE_ROOT),
    "--orig_dir", str(PUBLIC_ROOT),
    "--scene_name", SCENE,
    "--iterations", "30000",
]
print(" ".join(pixelgs_render_cmd))
subprocess.run(pixelgs_render_cmd, cwd=REPO, check=True)
pixelgs_rendered_files = [
    path for path in (PIXELGS_IMAGE_ROOT / SCENE).iterdir() if path.is_file()
]
print("So anh Pixel-GS render:", len(pixelgs_rendered_files))
assert len(pixelgs_rendered_files) == 60


In [ ]:
pixelgs_eval_cmd = [
    sys.executable, "eval_scene.py",
    "--input_dir", str(CLEAN_ROOT),
    "--image_dir", str(PIXELGS_IMAGE_ROOT),
    "--eval_dir", str(PIXELGS_EVAL_ROOT),
    "--scene_name", SCENE,
    "--psnr_max", "40",
]
subprocess.run(pixelgs_eval_cmd, cwd=REPO, check=True)
pixelgs_result_path = PIXELGS_EVAL_ROOT / f"{SCENE}.json"
with open(pixelgs_result_path, encoding="utf-8") as file:
    pixelgs_result = json.load(file)
assert pixelgs_result["num_images"] == 60
print(json.dumps(pixelgs_result, indent=2))
